In [1]:
from pathlib import Path
import shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Step 2: Define the dataset paths

In [2]:
PROJECT_DIR = Path("..")

SOURCE_DIR = PROJECT_DIR / "plantvillage" / "raw" / "color"
OUTPUT_DIR = PROJECT_DIR / "dataset" / "processed"

classes = [
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy"
]

print("Source directory exists:", SOURCE_DIR.exists())
print("Output directory:", OUTPUT_DIR.resolve())

Source directory exists: True
Output directory: C:\Users\Admin\Documents\Potato_Leaf_Disease_Project\dataset\processed


## Step 3: Collect image paths and labels

Create a DataFrame containing every image's file path and its corresponding disease class.

In [3]:
image_records = []

for class_name in classes:
    class_folder = SOURCE_DIR / class_name

    for image_path in class_folder.glob("*.JPG"):
        image_records.append({
            "image_path": str(image_path),
            "label": class_name
        })

df = pd.DataFrame(image_records)

print("Total images:", len(df))
print("\nImages per class:")
print(df["label"].value_counts())

display(df.head())

Total images: 2152

Images per class:
label
Potato___Early_blight    1000
Potato___Late_blight     1000
Potato___healthy          152
Name: count, dtype: int64


,image_path,label
0,..\plantvillage\raw\color\Potato___Early_bligh...,Potato___Early_blight
1,..\plantvillage\raw\color\Potato___Early_bligh...,Potato___Early_blight
2,..\plantvillage\raw\color\Potato___Early_bligh...,Potato___Early_blight
3,..\plantvillage\raw\color\Potato___Early_bligh...,Potato___Early_blight
4,..\plantvillage\raw\color\Potato___Early_bligh...,Potato___Early_blight


## Step 4: Split the dataset

We'll use:
- 70% training
- 15% validation
- 15% testing
We'll use stratify to preserve the class proportions in each split.


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))
print("Total images:", len(train_df) + len(val_df) + len(test_df))

Training images: 1506
Validation images: 323
Testing images: 323
Total images: 2152


## Step 5: Check class distribution in each split

Before copying images, let's confirm that stratification worked correctly

In [5]:
split_distribution = pd.DataFrame({
    "Training": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Testing": test_df["label"].value_counts()
}).fillna(0).astype(int)

display(split_distribution)

,Training,Validation,Testing
label,,,
Potato___Early_blight,700,150,150
Potato___Late_blight,700,150,150
Potato___healthy,106,23,23


Notice that the healthy class has only 106 training images. This confirms why we'll use class weighting during training.

## Step 6: Calculate class weights
We'll give more importance to the minority class so the model doesn't favour the two larger classes.

In [6]:
from sklearn.utils.class_weight import compute_class_weight

class_names = sorted(train_df["label"].unique())

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(class_names),
    y=train_df["label"]
)

class_weights = dict(enumerate(weights))

for index, weight in class_weights.items():
    print(f"{class_names[index]}: {weight:.3f}")

Potato___Early_blight: 0.717
Potato___Late_blight: 0.717
Potato___healthy: 4.736


Interpretation: The healthy class receives a much higher weight because it has fewer training examples. This encourages the model to pay more attention to healthy leaves during training.

## Step 7: Create the processed dataset folders
we'll organize the images into separate training, validation, and testing folders.

In [7]:
splits = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}

for split_name, split_df in splits.items():
    for class_name in classes:
        folder = OUTPUT_DIR / split_name / class_name
        folder.mkdir(parents=True, exist_ok=True)

print("Dataset folders created successfully.")

Dataset folders created successfully.


## Step 8: Copy images into their respective folders
we'll copy the images according to the splits we created. Your original PlantVillage images will remain untouched.

In [8]:
for split_name, split_df in splits.items():
    for _, row in split_df.iterrows():
        source_path = Path(row["image_path"])

        destination_path = (
            OUTPUT_DIR
            / split_name
            / row["label"]
            / source_path.name
        )

        shutil.copy2(source_path, destination_path)

print("All images copied successfully!")

All images copied successfully!


## Step 9: Verify the processed dataset
confirm that all images were copied correctly.

In [9]:
for split_name in ["train", "validation", "test"]:
    print(f"\n{split_name.upper()}")

    total = 0

    for class_name in classes:
        folder = OUTPUT_DIR / split_name / class_name
        count = len(list(folder.glob("*.JPG")))

        total += count
        print(f"{class_name}: {count}")

    print(f"Total: {total}")


TRAIN
Potato___Early_blight: 700
Potato___Late_blight: 700
Potato___healthy: 106
Total: 1506

VALIDATION
Potato___Early_blight: 150
Potato___Late_blight: 150
Potato___healthy: 23
Total: 323

TEST
Potato___Early_blight: 150
Potato___Late_blight: 150
Potato___healthy: 23
Total: 323


## Step 10: Save the split information

In [10]:
train_df.assign(split="train").to_csv(
    "../dataset/train_split.csv", index=False
)

val_df.assign(split="validation").to_csv(
    "../dataset/validation_split.csv", index=False
)

test_df.assign(split="test").to_csv(
    "../dataset/test_split.csv", index=False
)

print("Dataset split records saved successfully.")

Dataset split records saved successfully.
